# Kaggle: train, evaluate, and push to the ABSA model registry

**This is a template, not a runnable-as-is notebook.** It needs a GPU and
Kaggle context this repo's own dev environment doesn't have, plus two
things only you can fill in before it will do anything real:

1. **`HF_REPO_ID`** below — a Hugging Face Hub model repo you control
   (e.g. `"your-username/nlp-transformer-analysis-absa"`). The trained
   checkpoint gets pushed there; Neon only stores a pointer to it.
2. **An `HF_TOKEN` Kaggle secret** — Kaggle notebook Settings → Add-ons →
   Secrets → add a secret named `HF_TOKEN` with write access to that repo
   (from https://huggingface.co/settings/tokens).

## What this notebook does

Trains the main-cohort model (`src/train.py`'s `Trainer`, unchanged from
local runs), evaluates it (`src/evaluate.py`'s `evaluate_model`), runs it
over the same curated example set `app.py` shows in its "Example Gallery"
page, pushes the checkpoint to Hugging Face Hub, and writes a
`results.json` output artifact matching the contract
`scripts/load_absa_results.py` expects.

## After this notebook finishes

1. Download `results.json` from this notebook's Output panel.
2. Locally (where your `.env` has `DATABASE_URL`), run:
   ```bash
   python scripts/load_absa_results.py results.json
   ```
3. `streamlit run app.py` — the sidebar should now show
   `Loaded trained run ...` instead of the untrained-baseline warning.

## 1. Setup

Kaggle's Python images already have `torch`/`transformers`/`datasets`
preinstalled. `datasets` versions newer than what `McAuley-Lab/Amazon-
Reviews-2023` supports will hit the same loading-script error this repo's
own `src/data.py` already runs into locally — if that happens here, pin
an older `datasets` version, or point `config.yaml`'s `data.dataset_name`
/`dataset_subset` at a dataset that doesn't need `trust_remote_code`.

In [ ]:
!pip install -q huggingface_hub

## 2. Bring in this repo

Either upload `src/`, `config.yaml`, and `db/schema.sql` as a Kaggle
Dataset attached to this notebook, or clone the repo directly if it's
reachable from Kaggle (public, or private with a token):

In [ ]:
# Option A: repo already attached as a Kaggle Dataset at /kaggle/input/...
# import sys
# sys.path.insert(0, "/kaggle/input/nlp-transformer-analysis/src/..")

# Option B: clone directly
# !git clone https://github.com/<you>/NLPTransformerAnalysis.git /kaggle/working/repo
# import sys
# sys.path.insert(0, "/kaggle/working/repo")

import sys
from pathlib import Path

REPO_ROOT = Path(".")  # adjust to wherever src/ + config.yaml ended up above
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
from src.data import load_config, create_dataloaders
from src.model import build_model
from src.train import Trainer
from src.evaluate import evaluate_model
from src.inference import Predictor

config = load_config(str(REPO_ROOT / "config.yaml"))
# Optionally override for this run, e.g. a larger subset now that GPU time is available:
# config["data"]["train_size"] = 20000
# config["training"]["epochs"] = 5
config

## 3. Train

In [ ]:
dataloaders = create_dataloaders(config)
model = build_model(config)
trainer = Trainer(model, config, dataloaders["train"], dataloaders["val"])
history = trainer.train()
history["train_loss"][-1], history["val_loss"][-1]

## 4. Evaluate

Writes `results/metrics.json` too (via `Trainer`'s own checkpointing +
`main.py evaluate`'s pattern) — `metrics` below is copied verbatim into
the results JSON pushed to Neon, so there's no separate metric format to
maintain.

In [ ]:
checkpoint_path = str(Path(config["paths"]["model_dir"]) / "checkpoint_best.pt")
predictor = Predictor.from_checkpoint(checkpoint_path, config_path=str(REPO_ROOT / "config.yaml"))
metrics = evaluate_model(predictor, dataloaders["test"], config)
metrics["aspect"]["accuracy"], metrics["sentiment"]["accuracy"]

## 5. Run the example set

Duplicated from `app.py`'s `EXAMPLE_REVIEWS` rather than imported — this
kernel has no reason to install `streamlit`/`altair` just to reuse a
seven-item dict. Keep this in sync with `app.py` by hand if the examples
there change.

In [ ]:
EXAMPLE_REVIEWS = {
    "[Mixed] Great quality, slow shipping": "Great quality but shipping took 3 weeks",
    "[Mixed] Food vs. service": "The food was excellent but the waiter was rude and slow.",
    "[Positive] Glowing review": "Absolutely love this product! Great quality and fast shipping.",
    "[Negative] Broke fast": "Terrible experience. Broke after one week. Customer service was unhelpful.",
    "[Neutral] Middling": "Decent for the price. Nothing special but gets the job done.",
    "[Negative] Damaged shipment": "Package arrived damaged. Took 3 weeks to get here. Very disappointed.",
    "[Positive] Easy setup": "Easy to set up and use. Instructions were clear. Good value for money.",
}

example_keys = list(EXAMPLE_REVIEWS.keys())
example_results = predictor.predict_batch(list(EXAMPLE_REVIEWS.values()))
example_predictions = [
    {
        "example_key": key,
        "text": r["text"],
        "aspect": r["aspect"],
        "aspect_confidence": r["aspect_confidence"],
        "sentiment": r["sentiment"],
        "sentiment_confidence": r["sentiment_confidence"],
    }
    for key, r in zip(example_keys, example_results)
]
example_predictions[:2]

## 6. Push the checkpoint to Hugging Face Hub

**Fill in `HF_REPO_ID` before running this cell.**

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

HF_REPO_ID = "your-username/nlp-transformer-analysis-absa"  # <-- fill in

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
commit_info = api.upload_file(
    path_or_fileobj=checkpoint_path,
    path_in_repo="checkpoint_best.pt",
    repo_id=HF_REPO_ID,
)
hf_revision = commit_info.oid
hf_revision

## 7. Assemble `results.json`

This is the exact contract `scripts/load_absa_results.py` validates
against — see that file's `REQUIRED_KEYS`.

In [ ]:
import json
from datetime import datetime, timezone

model_version = f"bert-absa-{datetime.now(timezone.utc):%Y-%m-%d}-kaggle-gpu-01"  # bump the suffix per run

results = {
    "model_version": model_version,
    "track": "bert-absa",
    "run_timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "environment": "kaggle_gpu",
    "hf_repo_id": HF_REPO_ID,
    "hf_revision": hf_revision,
    "hf_filename": "checkpoint_best.pt",
    "data_config": config["data"],
    "hyperparameters": config["training"],
    "metrics": metrics,
    "example_predictions": example_predictions,
    "notes": "",
}

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Wrote /kaggle/working/results.json for {model_version}")

## 8. Next step

Download `/kaggle/working/results.json` from this notebook's Output
panel, then locally:

```bash
python scripts/load_absa_results.py results.json
```